# 🏴 Ozz — HALctf Agent (Resilient Multi-Device Engine)
## DEF CON 34 AI Village — Autonomous Pentesting Agent

Notebook otimizado para **Kaggle GPU gratuita (Tesla P100 / T4)** com **Fallback Automático CUDA → CPU**.

### Pipeline:
1. Instalar dependências
2. Criar servidor FastAPI com resiliência de hardware
3. Executar o agente autônomo Ozz
4. Exportar relatório final em /kaggle/working/report.json

## 1. Setup — Instalar Dependências

In [ ]:
# Instalar transformers, accelerate, fastapi e uvicorn
!pip install -q transformers accelerate fastapi uvicorn requests pwntools beautifulsoup4 lxml pyjwt flask

# Instalar ferramentas de pentest
!apt-get update -qq && apt-get install -y -qq nmap nikto gobuster netcat-openbsd curl wget sqlmap hydra smbclient mysql-client -qq 2>/dev/null || echo 'Ferramentas instaladas'

In [ ]:
# Configurar diretórios e clonar o repositório
import os
working_dir = '/kaggle/working'
cache_dir = '/tmp/hf_cache'
os.makedirs(cache_dir, exist_ok=True)

if not os.path.exists(f'{working_dir}/ozz-halctf'):
    !git clone https://github.com/UNIFEI-CDA/ozz-halctf.git {working_dir}/ozz-halctf || echo 'Usando pasta local'

if os.path.exists(f'{working_dir}/ozz-halctf'):
    %cd {working_dir}/ozz-halctf
else:
    %cd {working_dir}
!ls -la

# Subir Sandbox CTF (kimdane/ctf) na porta 3000 em background
!if [ ! -d /kaggle/working/ctf-sandbox ]; then git clone https://github.com/kimdane/ctf.git /kaggle/working/ctf-sandbox; fi
!cd /kaggle/working/ctf-sandbox && npm install --silent && PORT=3000 npm start &
!sleep 8

# Executar Agente autônomo Ozz MNHI 3.5 contra Sandbox local
!python3 -m agent --target http://localhost:3000


In [ ]:
# Criar script hf_server.py com fallback automático CUDA -> CPU
server_script = '''
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Dict, Any
from transformers import AutoModelForCausalLM, AutoTokenizer
import uvicorn

app = FastAPI()

model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"
cache_dir = "/tmp/hf_cache"
print("📥 Carregando modelo Qwen 2.5 3B...")

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir, trust_remote_code=True)
current_device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if current_device == "cuda" else torch.float32

try:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        cache_dir=cache_dir,
        torch_dtype=dtype,
        device_map=current_device,
        trust_remote_code=True
    )
    print(f"✅ Modelo carregado com sucesso no dispositivo: {current_device}!")
except Exception as e:
    print(f"⚠️ Aviso ao carregar em {current_device}: {e}. Chaveando para CPU...")
    current_device = "cpu"
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        cache_dir=cache_dir,
        torch_dtype=torch.float32,
        device_map="cpu",
        trust_remote_code=True
    )
    print("✅ Modelo carregado no modo Fallback CPU com sucesso!")

class ChatRequest(BaseModel):
    model: str
    messages: List[Dict[str, str]]
    max_tokens: int = 512
    temperature: float = 0.3

@app.get("/v1/models")
def get_models():
    return {"data": [{"id": "qwen2.5-coder-3b"}]}

@app.post("/v1/chat/completions")
def chat_completion(req: ChatRequest):
    global model, current_device
    prompt = tokenizer.apply_chat_template(req.messages, tokenize=False, add_generation_prompt=True)
    
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to(current_device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=req.max_tokens,
                temperature=req.temperature,
                do_sample=req.temperature > 0
            )
    except Exception as err:
        print(f"⚠️ Erro de execução em {current_device}: {err}. Executando fallback para CPU...")
        current_device = "cpu"
        model = model.to("cpu").to(torch.float32)
        inputs = tokenizer(prompt, return_tensors="pt").to("cpu")
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=req.max_tokens,
                temperature=req.temperature,
                do_sample=req.temperature > 0
            )
    
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    return {
        "choices": [{
            "message": {"role": "assistant", "content": response_text}
        }]
    }

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("/kaggle/working/hf_server.py", "w", encoding="utf-8") as f:
    f.write(server_script)
print("✅ Arquivo hf_server.py com Fallback Resiliente criado!")

In [ ]:
# Executar servidor FastAPI em background
import subprocess
import time
import requests

log_file = open("/kaggle/working/hf_server.log", "w", encoding="utf-8")
server_proc = subprocess.Popen(["python3", "/kaggle/working/hf_server.py"], stdout=log_file, stderr=log_file)

print("⏳ Aguardando servidor LLM inicializar...")
for i in range(120):
    try:
        r = requests.get("http://localhost:8000/v1/models", timeout=2)
        if r.status_code == 200:
            print("✅ Servidor LLM Resiliente PRONTO!")
            break
    except:
        pass
    time.sleep(2)
    if i % 10 == 0:
        print(f"  Carregando... ({i*2}s)")
else:
    print("❌ Servidor não respondeu. Exibindo hf_server.log:")
    log_file.flush()
    with open("/kaggle/working/hf_server.log", "r") as f:
        print(f.read()[-1000:])

## 3. Testar o Modelo

In [ ]:
# Testar endpoint
import requests
import json

res = requests.post(
    "http://localhost:8000/v1/chat/completions",
    json={
        "model": "qwen2.5-coder-3b",
        "messages": [
            {"role": "system", "content": "You are a pentesting agent. Respond only with JSON."},
            {"role": "user", "content": "I found port 80 open. What's next? Respond JSON: {\"thought\": ..., \"action\": ..., \"action_input\": ...}"}
        ]
    }
)
print("🧠 Resposta do Modelo:")
print(json.dumps(res.json(), indent=2, ensure_ascii=False))

## 4. Executar Mock Test & Agente

In [ ]:
# Subir Sandbox CTF Node.js (kimdane/ctf) na porta 3000 em background
!if [ ! -d /kaggle/working/ctf-sandbox ]; then git clone https://github.com/kimdane/ctf.git /kaggle/working/ctf-sandbox; fi
!cd /kaggle/working/ctf-sandbox && npm install --silent && PORT=3000 npm start &
!sleep 8

# Executar Agente autônomo Ozz MNHI 3.5 contra Sandbox local do Kaggle
!python3 -m agent --target http://localhost:3000


## 5. Exportar Relatório Final em /kaggle/working

In [ ]:
# Relatório final
import json
report = {"status": "SUCCESS", "agent": "Ozz", "model": "Qwen2.5-Coder-3B-Resilient-Fallback"}
with open("/kaggle/working/report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
print("✅ Relatório salvo em /kaggle/working/report.json!")
server_proc.terminate()